# Exercise 05 : Two Agents over A2A

Open `exercises/exercise05-two-agents-over-a2a.ipynb` and run the setup cell. Build a service desk that calls a separately served parts agent over A2A by replacing the `____` blanks. The solution is `solutions/exercise05-two-agents-over-a2a.ipynb`.

Worked examples to revisit : module 06 demo03 · served-agents, demo04 · two-agents-over-a2a.

Made up, not from work. Use the supplied Aurelia data and fictional policy values.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


In [2]:
import json
import operator
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated, TypedDict
from uuid import UUID, uuid4

from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command, interrupt

class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for batch in response.generations:
                for generation in batch:
                    usage = generation.message.usage_metadata or {}
                    self.input_tokens += usage.get("input_tokens", 0)
                    self.output_tokens += usage.get("output_tokens", 0)

counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=650, max_retries=0)
started = time.perf_counter()
DB = Path(data("capstone/service.db")).resolve()
LIMIT = {"recursion_limit": 24}

def part_facts(part_no):
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        db.row_factory = sqlite3.Row
        row = db.execute("SELECT part_no, name, stock, price_eur, depot FROM parts WHERE part_no = ?",
                         (part_no,)).fetchone()
    if row is None:
        raise ValueError("Unknown Aurelia part")
    return dict(row)

def check(label, condition):
    print("check :", label, "PASS" if condition else "FAIL")
    assert condition, label

def finish_metrics(slug, server_usage=None):
    rows = server_usage or []
    result = {"model": MODEL, "rps": float(os.environ.get("COURSE_RPS", "2")),
              "calls": counter.calls + len(rows),
              "input_tokens": counter.input_tokens + sum(r.get("input_tokens", 0) for r in rows),
              "output_tokens": counter.output_tokens + sum(r.get("output_tokens", 0) for r in rows),
              "seconds": round(time.perf_counter() - started, 1)}
    print("metrics :", json.dumps(result))
    Path(slug + "-metrics.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")

## Step 1 : Graph sources

Complete the coordinator source below. Run each source-writing cell before starting the servers. Agent Server needs the `server` dependency group. It runs on the classroom laptop and in a Codespace, and not on Colab.

In [3]:
import atexit
import socket
import subprocess
import httpx

servers = []

def stop_servers():
    for process in reversed(servers):
        if process.poll() is None:
            if os.name == "nt":
                subprocess.run(["taskkill", "/PID", str(process.pid), "/T", "/F"],
                               capture_output=True, check=False)
            else:
                os.killpg(process.pid, 15)
            process.wait(timeout=20)

atexit.register(stop_servers)

def start_server(port, graph_id, folder):
    if port not in {2124, 2125}:
        raise ValueError("Use only ports 2124 and 2125")
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", port))
    config = folder / "langgraph.json"
    config.write_text(json.dumps({"dependencies": ["."], "graphs": {graph_id: "./graph.py:graph"}}), encoding="utf-8")
    env = os.environ.copy()
    env["COURSE_RPS"] = os.environ.get("COURSE_RPS", "1")
    env["PYTHONUTF8"] = "1"
    env["LANGSMITH_TRACING"] = "false"
    log = (folder / "server.log").open("w", encoding="utf-8")
    process = subprocess.Popen(["uv", "run", "langgraph", "dev", "--port", str(port),
                                "--config", str(config), "--no-browser", "--no-reload", "--allow-blocking"],
                               cwd=folder, env=env, stdout=log, stderr=subprocess.STDOUT,
                               creationflags=subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0,
                               start_new_session=os.name != "nt")
    servers.append(process)
    url = f"http://127.0.0.1:{port}"
    deadline = time.monotonic() + 90
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError("Agent Server exited. Read " + str(folder / "server.log"))
        try:
            if httpx.get(url + "/ok", timeout=3, trust_env=False).status_code == 200:
                return url
        except httpx.TransportError:
            pass
        time.sleep(0.5)
    raise TimeoutError("Agent Server readiness timed out")

def assistant_for(url, graph_id):
    response = httpx.post(url + "/assistants/search", json={"limit": 20}, trust_env=False, timeout=30)
    response.raise_for_status()
    return next(a["assistant_id"] for a in response.json() if a["graph_id"] == graph_id)

def rpc(url, assistant_id, method, params):
    response = httpx.post(url + "/a2a/" + assistant_id, trust_env=False, timeout=240,
                          headers={"Accept": "application/json"},
                          json={"jsonrpc": "2.0", "id": str(uuid4()), "method": method, "params": params})
    response.raise_for_status()
    body = response.json()
    if "error" in body:
        raise RuntimeError(json.dumps(body["error"]))
    return body["result"]

def send_params(text, context):
    return {"message": {"role": "ROLE_USER", "messageId": str(uuid4()), "contextId": context,
                        "parts": [{"text": text}]}, "configuration": {"historyScope": "task"}}

def task_text(task):
    return "\n".join(p["text"] for a in task.get("artifacts", []) for p in a["parts"] if "text" in p)

def thread_state(url, context):
    response = httpx.get(url + "/threads/" + context + "/state", trust_env=False, timeout=30)
    response.raise_for_status()
    return response.json()["values"]

In [4]:
server_prelude = r'''
import json
import operator
import os
import runpy
import sqlite3
import time
from pathlib import Path
from typing import Annotated
from uuid import uuid4

import httpx
from langchain.messages import SystemMessage
from langgraph.graph import END, START, MessagesState, StateGraph

ROOT = next(p for p in Path(__file__).resolve().parents if (p / "pyproject.toml").exists())
setup = runpy.run_path(str(ROOT / "build" / "setup_cell.py"))
llm = setup["make_llm"](max_tokens=500, max_retries=0)

class ServedState(MessagesState):
    usage_history: Annotated[list[dict], operator.add]
    exchange: dict
'''

In [5]:
specialist_source = server_prelude + r'''
def answer(state):
    database = ROOT / "data/capstone/service.db"
    with sqlite3.connect(f"file:{database.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute("SELECT part_no, stock, depot FROM parts WHERE part_no IN ('CT-FN-02', 'CT-SR-03')").fetchall()
    reply = llm.invoke([SystemMessage(
        "Report Aurelia stock from these facts only. Keep part numbers, counts and depots. "
        "Do not reserve anything.\n" + json.dumps(rows)), *state["messages"]])
    return {"messages": [reply], "usage_history": [reply.usage_metadata or {}]}

b = StateGraph(ServedState)
b.add_node("answer", answer)
b.add_edge(START, "answer")
b.add_edge("answer", END)
graph = b.compile()
'''

In [6]:
coordinator_source = server_prelude + r'''
def coordinate(state, config):
    context = str(config["configurable"]["thread_id"])
    time.sleep(1 / float(os.environ["COURSE_RPS"]))
    with httpx.Client(timeout=240, trust_env=False) as client:
        search = client.post("http://127.0.0.1:2125/assistants/search", json={"limit": 20})
        search.raise_for_status()
        specialist_id = next(a["assistant_id"] for a in search.json() if a["graph_id"] == "aurelia_parts")
        request = {"jsonrpc": "2.0", "id": str(uuid4()), "method": "SendMessage", "params": {
            "message": {"role": "ROLE_USER", "messageId": str(uuid4()), "contextId": context,
                        "parts": [{"text": str(state["messages"][-1].content)}]},
            "configuration": {"historyScope": "task"}}}
        response = client.post("http://127.0.0.1:2125/a2a/" + specialist_id,
                               json=request, headers={"Accept": "application/json"})
        response.raise_for_status()
    body = response.json()
    if "error" in body:
        raise RuntimeError("Specialist A2A error : " + json.dumps(body["error"]))
    if "task" not in body["result"]:
        raise RuntimeError("SendMessage returns result.task ; this result has keys "
                           + str(sorted(body["result"])) + ". The legacy message/send unwraps the task.")
    task = body["result"]["task"]
    if task["status"]["state"] != "TASK_STATE_COMPLETED":
        raise RuntimeError("Specialist task did not complete : " + json.dumps(task["status"]))
    evidence = "\n".join(p["text"] for a in task.get("artifacts", []) for p in a["parts"] if "text" in p)
    time.sleep(1 / float(os.environ["COURSE_RPS"]))
    reply = llm.invoke([SystemMessage("Write an Aurelia service handover from the specialist evidence. "
                                     "Preserve counts and depots. No order was placed.\n" + evidence),
                        *state["messages"]])
    return {"messages": [reply], "usage_history": [reply.usage_metadata or {}],
            "exchange": {"request": request, "remote_task_id": task["id"],
                         "remote_context_id": task["contextId"], "evidence": evidence}}

b = StateGraph(ServedState)
b.add_node("coordinate", coordinate)
b.add_edge(START, "coordinate")
b.add_edge("coordinate", END)
graph = b.compile()
'''

In [7]:
if ROOT is None:
    raise RuntimeError("Run this server exercise in the local course repository")
runtime = Path.cwd() / "exercise05-two-agents-over-a2a" / "workspace"
parts_folder, desk_folder = runtime / "parts", runtime / "desk"
for folder, source in [(parts_folder, specialist_source), (desk_folder, coordinator_source)]:
    compile(source, "graph.py", "exec")
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "graph.py").write_text(source, encoding="utf-8")
check("edited sources written", all((p / "graph.py").exists() for p in [parts_folder, desk_folder]))

check : edited sources written PASS


## Step 2 : Agent cards

Complete the card query. The UUID comes from `/assistants/search`. Inspect the printed interface before sending any task.

In [8]:
try:
    parts_url = start_server(2125, "aurelia_parts", parts_folder)
    desk_url = start_server(2124, "aurelia_desk", desk_folder)
    parts_id = assistant_for(parts_url, "aurelia_parts")
    desk_id = assistant_for(desk_url, "aurelia_desk")
    cards = []
    for url, assistant_id in [(parts_url, parts_id), (desk_url, desk_id)]:
        response = httpx.get(url + "/.well-known/agent-card.json",
                             params={"assistant_id": assistant_id}, trust_env=False, timeout=30)
        response.raise_for_status()
        card = response.json()
        cards.append(card)
        print("card :", card["name"], card["supportedInterfaces"])
        UUID(assistant_id)
except Exception:
    stop_servers()
    raise
check("both cards advertise JSONRPC", all(c["supportedInterfaces"][0]["protocolBinding"] == "JSONRPC" for c in cards))

card : aurelia_parts [{'url': 'http://127.0.0.1:2125/a2a/211d5806-1f67-5dad-89d8-038d3a503e0a', 'protocolBinding': 'JSONRPC', 'protocolVersion': '1.0'}]
card : aurelia_desk [{'url': 'http://127.0.0.1:2124/a2a/fe1029a9-0d7b-5751-b9a8-21621dd4d05d', 'protocolBinding': 'JSONRPC', 'protocolVersion': '1.0'}]
check : both cards advertise JSONRPC PASS


## Step 3 : First task

Fill the context constructor, method and result key. Read the stored exchange to confirm that the desk itself called the parts agent.

In [9]:
try:
    context = str(uuid4())
    first = rpc(desk_url, desk_id, "SendMessage", send_params(
        "Report CT-FN-02 and CT-SR-03 stock and depots for an Aurelia CT-400 visit.", context))["task"]
    print("desk task state :", first["status"]["state"])
    if first["status"]["state"] != "TASK_STATE_COMPLETED":
        raise RuntimeError("The desk task failed : " + json.dumps(first["status"].get("message"))
                           + ". Read " + str(desk_folder / "server.log"))
    first_state = thread_state(desk_url, context)
    first_exchange = first_state["exchange"]
    print("handover :", task_text(first))
    print("remote exchange :", first_exchange)
    print("task ids :", first["id"], first_exchange["remote_task_id"])
except Exception:
    stop_servers()
    raise
check("completed with text", first["status"]["state"] == "TASK_STATE_COMPLETED" and bool(task_text(first)))
check("two tasks share context", first["contextId"] == first_exchange["remote_context_id"] == context
      and first["id"] != first_exchange["remote_task_id"])
check("stock evidence", all(p in first_exchange["evidence"] for p in ["CT-FN-02", "CT-SR-03", "Houten"]))
sent = first_exchange["request"]["params"]["message"]
check("A2A 1.0 message", sent["role"] == "ROLE_USER" and all("kind" not in p for p in sent["parts"]))

desk task state : TASK_STATE_COMPLETED
handover : **Aurelia CT-400 visit stock**
- **CT-FN-02:** 6 at Houten
- **CT-SR-03:** 0 at Houten

Nothing was reserved, and no order was placed.
remote exchange : {'request': {'jsonrpc': '2.0', 'id': '76f8ac53-2b40-44df-984a-8573176b72d2', 'method': 'SendMessage', 'params': {'message': {'role': 'ROLE_USER', 'messageId': 'bc7a0ac4-bf4f-4263-b4ed-7d76fd276f4c', 'contextId': 'd8497754-d08f-4c17-8f09-00882e345f1f', 'parts': [{'text': 'Report CT-FN-02 and CT-SR-03 stock and depots for an Aurelia CT-400 visit.'}]}, 'configuration': {'historyScope': 'task'}}}, 'remote_task_id': 'd8497754-d08f-4c17-8f09-00882e345f1f:01a17b68-afdc-7a93-a51a-db5944e38e0a', 'remote_context_id': 'd8497754-d08f-4c17-8f09-00882e345f1f', 'evidence': 'Aurelia CT-400 visit stock:\n- CT-FN-02: 6 at Houten\n- CT-SR-03: 0 at Houten\n\nNothing reserved.'}
task ids : d8497754-d08f-4c17-8f09-00882e345f1f:01a17b68-7d35-7be2-8c17-27c854775ce5 d8497754-d08f-4c17-8f09-00882e345f1f:01a17b68

## Step 4 : Same context

Keep `context` and request a new task. Compare the answer artifacts with the keys returned by `GetTask`. `historyScope="task"` filters the response history for the requested task. It does not trim model input. `historyScope` is an Agent Server extension ; the A2A 1.0 field is `historyLength`, and another A2A server will ignore `historyScope`. The `kind` key in the response is a leftover from A2A 0.3 that Agent Server still sends.

In [10]:
try:
    follow = rpc(desk_url, desk_id, "SendMessage", send_params("Which of those two parts is out of stock?", context))["task"]
    fetched = rpc(desk_url, desk_id, "GetTask", {"id": follow["id"], "historyScope": "task"})
    whole = rpc(desk_url, desk_id, "GetTask", {"id": follow["id"]})
    desk_state = thread_state(desk_url, context)
    parts_state = thread_state(parts_url, context)
    server_usage = desk_state["usage_history"] + parts_state["usage_history"]
    print("follow-up :", task_text(follow))
    print("GetTask keys :", sorted(fetched))
    print("history lengths :", len(fetched.get("history", [])), len(whole.get("history", [])))
    print("server usage :", server_usage)
except Exception:
    stop_servers()
    raise
check("new task in same context", follow["id"] != first["id"] and follow["contextId"] == context)
check("GetTask is unwrapped", fetched["id"] == follow["id"] and "artifacts" not in fetched)
check("task history selects follow-up", len(fetched.get("history", [])) < len(whole.get("history", [])))
check("follow-up identifies unavailable part", "CT-SR-03" in task_text(follow))
check("all served calls counted", len(server_usage) == 4 and all(r.get("total_tokens", 0) > 0 for r in server_usage))

follow-up : CT-SR-03 is out of stock at Houten.
GetTask keys : ['contextId', 'history', 'id', 'kind', 'status']
history lengths : 2 4
server usage : [{'input_tokens': 96, 'output_tokens': 52, 'total_tokens': 148, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, {'input_tokens': 141, 'output_tokens': 17, 'total_tokens': 158, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, {'input_tokens': 87, 'output_tokens': 42, 'total_tokens': 129, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, {'input_tokens': 145, 'output_tokens': 19, 'total_tokens': 164, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}]
check : new task in same context PASS
check : GetTask is unwrapped PASS
chec

## Step 5 : Shutdown

Run the final cell after the optional work. It stops only the servers this notebook started. Server calls are counted from the persisted usage lists on both servers.

### If time permits

- Send a third question through the desk with the same context and compare both histories
- Call the parts agent with `message/send` and compare its response shape with `SendMessage`
- Send a context that is not a UUID and read the JSON-RPC error ; no model call is made

In [11]:
stop_servers()
check("both servers stopped", len(servers) == 2 and all(p.poll() is not None for p in servers))
for port in [2124, 2125]:
    with socket.socket() as probe:
        check(f"port {port} released", probe.connect_ex(("127.0.0.1", port)) != 0)
finish_metrics("exercise05-two-agents-over-a2a", server_usage)

check : both servers stopped PASS


check : port 2124 released PASS


check : port 2125 released PASS
metrics : {"model": "openai/gpt-6-luna", "rps": 0.4, "calls": 4, "input_tokens": 469, "output_tokens": 130, "seconds": 39.2}
